# Logistic Regression

### X and Y

In [1]:
import pandas as pd

# Load CSV
features_df = pd.read_csv("hyperaktiv_with_controls/features.csv", sep=";")

# Set participant ID as index
features_df = features_df.set_index("ID")

# This is your X matrix
X = features_df.copy()


In [2]:
print(X.shape)
print(X.isna().sum().sum(), "NaNs total")
print((X.var() == 0).sum(), "zero-variance features")

(116, 787)
696 NaNs total
47 zero-variance features


In [3]:
# Drop columns with any NaNs
X = X.dropna(axis=1)

# Drop zero-variance features
X = X.loc[:, X.var() > 0]

# Optional: reduce precision to save memory
# X = X.astype("float32")

In [4]:
import pandas as pd

# Load patient info
info = pd.read_csv("hyperaktiv_with_controls/patient_info.csv", sep=";")

# Keep only relevant columns
info = info[["ID", "ADHD", "ADD"]]

# Align with X (important!)
info = info.set_index("ID").loc[X.index]

# Create multiclass target
def build_multiclass_label(row):
    if row["ADHD"] == 0:
        return 0  # No ADHD
    elif row["ADHD"] == 1 and row["ADD"] == 0:
        return 1  # ADHD without inattentive subtype
    elif row["ADHD"] == 1 and row["ADD"] == 1:
        return 2  # ADHD with inattentive subtype

y = info.apply(build_multiclass_label, axis=1)

# Convert to integer type
y = y.astype(int)


In [5]:
print(y.value_counts())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Index aligned:", X.index.equals(y.index))

0    71
2    23
1    22
Name: count, dtype: int64
X shape: (116, 734)
y shape: (116,)
Index aligned: True


In [6]:
from tsfresh.feature_selection import select_features

X_selected = select_features(
    X,
    y,
    fdr_level=0.05
)

print("Original features:", X.shape[1])
print("Selected features:", X_selected.shape[1])

Original features: 734
Selected features: 52


### Split Data

In [7]:
from sklearn.model_selection import train_test_split

# 80/20 split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=5,
    stratify=y
)

print(f"Train/Val samples: {len(X_train_val)}")
print(f"Test samples: {len(X_test)}")


Train/Val samples: 92
Test samples: 24


### Logistic Regression

In [8]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

C_values = [0.01, 0.1, 1, 10]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)

results = []
reports = []

for C in C_values:
    fold_idx = 0

    for train_idx, val_idx in skf.split(X_train_val, y_train_val):
        fold_idx += 1

        X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
        y_train, y_val = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(
                C=C,
                penalty="l2",
                solver="lbfgs",
                class_weight="balanced",
                multi_class="multinomial",
                max_iter=1000,
                random_state=5
            ))
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

        f1 = f1_score(y_val, y_pred, average="macro")

        results.append({
            "C": C,
            "fold": fold_idx,
            "f1_macro": f1
        })

        reports.append({
            "C": C,
            "fold": fold_idx,
            "report": classification_report(y_val, y_pred, output_dict=True)
        })


results_df = pd.DataFrame(results)

summary = (
    results_df
    .groupby("C")["f1_macro"]
    .agg(["mean", "std"])
    .reset_index()
)

summary

c:\Users\gur15\anaconda3\envs\syde577\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\gur15\anaconda3\envs\syde577\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\gur15\anaconda3\envs\syde577\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\gur15\anaconda3\envs\syde577\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: F

,C,mean,std
0,0.01,0.479350,0.073517
1,0.10,0.497918,0.099127
2,1.00,0.465307,0.027940
3,10.00,0.430242,0.036959


### Best Parameters

In [9]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

BEST_C = 1   # best C

final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=BEST_C,
        penalty="l2",
        solver="lbfgs",
        class_weight="balanced",
        multi_class="multinomial",
        max_iter=1000,
        random_state=5
    ))
])

final_pipeline.fit(X_train_val, y_train_val)

y_test_pred = final_pipeline.predict(X_test)

final_f1_macro = f1_score(y_test, y_test_pred, average="macro")
final_bal_acc = balanced_accuracy_score(y_test, y_test_pred)

print("Final Test Macro F1:", final_f1_macro)
print("Final Test Balanced Accuracy:", final_bal_acc)

print("\nFinal Test Classification Report:")
print(classification_report(y_test, y_test_pred))

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Final Test Macro F1: 0.46654877689360447
Final Test Balanced Accuracy: 0.4611111111111111

Final Test Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.73      0.76        15
           1       0.50      0.25      0.33         4
           2       0.25      0.40      0.31         5

    accuracy                           0.58        24
   macro avg       0.51      0.46      0.47        24
weighted avg       0.63      0.58      0.59        24


Final Test Confusion Matrix:
[[11  0  4]
 [ 1  1  2]
 [ 2  1  2]]


c:\Users\gur15\anaconda3\envs\syde577\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
